# AI Resume Ranker — Interactive NLP & Matching Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sami2515/ai_resume_ranker/blob/main/notebooks/02_resume_ranking_pipeline.ipynb)

**TechWiz — AI and Machine Learning Mania | Smart Resume Ranker for Recruiters**

**Institute:** Aptech Learning - North Nazimabad  
**Team Members:** Saba Noor • Muhmmad Sami • Ghanyan • Sami UR Rehman

This notebook demonstrates the main NLP ranking workflow: Ingesting resumes, extracting structured profiles, parsing Job Descriptions (JDs), computing hybrid Keyword + Semantic similarity scores, and generating explainability breakdowns.

In [ ]:
# 1. Setup Environment (Supports both Local Jupyter Notebook and Google Colab)
import sys
import os
import time
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab environment. Installing dependencies...")
    if not os.path.exists("ai_resume_ranker") and not os.path.exists("backend"):
        !git clone https://github.com/sami2515/ai_resume_ranker.git
        if os.path.exists("ai_resume_ranker"):
            %cd ai_resume_ranker
    !pip install -q spacy python-docx nltk scikit-learn sentence-transformers
    !python -m spacy download en_core_web_md
    !python -c "import nltk; nltk.download('stopwords'); nltk.download('wordnet'); nltk.download('punkt'); nltk.download('punkt_tab')"
    REPO_ROOT = Path.cwd()
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO_ROOT / "backend"))

from nlp_pipeline import parse_resume, extract_profile, rank_candidates, semantic_backend_name
from nlp_pipeline.jd_parser import parse_job_description
from nlp_pipeline.parser import ResumeParseError

print(f"Active Semantic Backend: {semantic_backend_name()}")
DATASET_DIR = REPO_ROOT / "datasets" / "resumes"
print(f"Dataset Directory: {DATASET_DIR}")

In [ ]:
# 2. Define Sample Job Description (JD)
JD_TITLE = "Senior Business Analyst"
JD_TEXT = """
We are seeking a Senior Business Analyst with 5+ years of experience.
Key responsibilities include requirements gathering, stakeholder management, business process modeling, use case documentation, and sprint planning.
Essential skills: SQL, Agile/Scrum, JIRA, Business Analysis, Process Flow, Stakeholder Management.
Certifications like CBAP or PMP are highly desirable.
"""

jd = parse_job_description(JD_TITLE, JD_TEXT)
print(f"Parsed JD Title: {jd.title}")
print(f"Required Skills ({len(jd.required_skills)}): {jd.required_skills}")
print(f"Minimum Experience: {jd.min_experience} years")

In [ ]:
# 3. Parse Sample Resumes from Dataset
resume_files = sorted(DATASET_DIR.glob("*.docx"))[:25] # Test sample of 25 resumes
print(f"Parsing {len(resume_files)} resumes...")

profiles = []
for f in resume_files:
    try:
        res = parse_resume(f)
        prof = extract_profile(res)
        profiles.append(prof)
    except ResumeParseError as e:
        print(f"Skipping {f.name}: {e}")

print(f"Successfully extracted {len(profiles)} candidate profiles.")

In [ ]:
# 4. Execute Hybrid Matching Engine & Ranking
t0 = time.time()
ranked_results = rank_candidates(profiles, jd)
elapsed = time.time() - t0

print(f"Ranking completed in {elapsed:.2f}s ({elapsed/len(profiles):.3f}s per resume)")
print("="*80)
print(f"{'Rank':<5}{'Candidate Name / ID':<26}{'Composite':<12}{'Keyword':<10}{'Semantic':<10}{'Confidence'}")
print("="*80)
for i, r in enumerate(ranked_results[:10], 1):
    cname = (r.candidate_name or r.candidate_filename)[:24]
    print(f"{i:<5}{cname:<26}{r.composite_score:<12.1f}{r.keyword_score:<10.1f}{r.semantic_score:<10.1f}{r.confidence}")
print("="*80)

In [ ]:
# 5. Explainability Breakdown for Top Ranked Candidate
top_candidate = ranked_results[0]
print(f"EXPLAINABILITY REPORT FOR TOP CANDIDATE: {top_candidate.candidate_name or top_candidate.candidate_filename}")
print("-"*60)
print(f"Composite Score:     {top_candidate.composite_score} / 100  (Confidence: {top_candidate.confidence})")
print(f"Keyword Match Score: {top_candidate.keyword_score}%")
print(f"Semantic Sim Score:  {top_candidate.semantic_score}%")
print(f"Matched Skills:      {top_candidate.matched_skills}")
print(f"Missing Skills:      {top_candidate.missing_skills}")
print(f"Estimated Experience:{top_candidate.experience_years} years")